<a href="https://colab.research.google.com/github/Lyv-ux/DI_Bootcamp/blob/main/W7D1_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# DAILY CHALLENGE - TEXT ANALYSIS OF BOOKS USING WORD CLOUD
# Lewis Carroll Books
# ============================================================

# ----------------------------
# STEP 1 - Import libraries
# ----------------------------
import re
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk import pos_tag
from nltk.chunk import ne_chunk

import spacy

from wordcloud import WordCloud

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# ----------------------------
# STEP 2 - Download resources
# ----------------------------
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# ----------------------------
# STEP 3 - Define URLs
# ----------------------------
urls = [
    "https://www.gutenberg.org/files/11/11-0.txt",
    "https://www.gutenberg.org/files/12/12-0.txt",
    "https://www.gutenberg.org/files/29042/29042-0.txt"
]

book_names = [
    "Alice in Wonderland",
    "Through the Looking Glass",
    "A Tangled Tale"
]

# ----------------------------
# STEP 4 - Load and clean texts
# ----------------------------
def load_texts(urls):

    corpus = []

    for url in urls:

        text = requests.get(url).text

        start = text.find("START")
        end = text.find("END")

        if start != -1 and end != -1:
            text = text[start:end]

        # keep letters and spaces only
        text = re.sub(r"[^a-zA-Z\s]", " ", text)

        # remove extra spaces
        text = re.sub(r"\s+", " ", text)

        corpus.append(text.lower())

    return corpus

corpus = load_texts(urls)

# ----------------------------
# STEP 5 - Print first 200 chars
# ----------------------------
print("\nFIRST 200 CHARACTERS\n")

for name, text in zip(book_names, corpus):
    print(f"\n{name}")
    print(text[:200])

# ----------------------------
# STEP 6 - Tokenization
# ----------------------------
tokenized_books = []

print("\nFIRST 150 TOKENS\n")

for name, text in zip(book_names, corpus):

    tokens = word_tokenize(text)

    tokenized_books.append(tokens)

    print(f"\n{name}")
    print(tokens[:150])

# ----------------------------
# STEP 7 - Remove stopwords
# ----------------------------
stop_words = set(stopwords.words("english"))

filtered_books = []

for tokens in tokenized_books:

    filtered = [
        word
        for word in tokens
        if word not in stop_words
    ]

    filtered_books.append(filtered)

# Check stopword removal
print("\nSTOPWORD CHECK\n")

for stopword in ["i", "me", "my", "myself", "we"]:

    total = sum(book.count(stopword) for book in filtered_books)

    print(f"{stopword}: {total}")

# ----------------------------
# STEP 8 - Stemming
# ----------------------------
stemmer = PorterStemmer()

stemmed_books = []

print("\nFIRST 50 STEMMED TOKENS\n")

for name, tokens in zip(book_names, filtered_books):

    stemmed = [stemmer.stem(word) for word in tokens]

    stemmed_books.append(stemmed)

    print(f"\n{name}")
    print(stemmed[:50])

# ----------------------------
# STEP 9 - Lemmatization
# ----------------------------
lemmatized_books = []

print("\nFIRST 50 LEMMATIZED TOKENS\n")

for name, tokens in zip(book_names, filtered_books):

    doc = nlp(" ".join(tokens))

    lemmas = [token.lemma_ for token in doc]

    lemmatized_books.append(lemmas)

    print(f"\n{name}")
    print(lemmas[:50])

# ----------------------------
# STEP 10 - POS TAGGING
# ----------------------------
print("\nPOS TAGS (FIRST 50)\n")

for name, tokens in zip(book_names, filtered_books):

    tags = pos_tag(tokens[:50])

    print(f"\n{name}")
    print(tags)

# ----------------------------
# STEP 11 - NAMED ENTITIES
# ----------------------------
print("\nNAMED ENTITIES\n")

for name, tokens in zip(book_names, filtered_books):

    tagged = pos_tag(tokens[:200])

    entities = ne_chunk(tagged)

    print(f"\n{name}")
    print(entities)

# ----------------------------
# STEP 12 - Word Clouds
# ----------------------------
for name, words in zip(book_names, lemmatized_books):

    text = " ".join(words)

    wc = WordCloud(
        width=1000,
        height=500,
        background_color="white"
    ).generate(text)

    plt.figure(figsize=(12,6))
    plt.imshow(wc)
    plt.axis("off")
    plt.title(name)
    plt.show()

# ----------------------------
# STEP 13 - Bag of Words
# Using lemmatized text
# ----------------------------
processed_docs = [
    " ".join(book)
    for book in lemmatized_books
]

vectorizer = CountVectorizer()

bow = vectorizer.fit_transform(processed_docs)

words = vectorizer.get_feature_names_out()

frequencies = np.asarray(bow.sum(axis=0)).flatten()

top_indices = frequencies.argsort()[::-1][:5]

print("\nTOP 5 MOST FREQUENT WORDS (ALL BOOKS)\n")

for idx in top_indices:

    print(words[idx], frequencies[idx])

# ----------------------------
# STEP 14 - Print BoW
# ----------------------------
print("\nBOW REPRESENTATION\n")

for doc_idx, doc in enumerate(bow):

    print(f"\nDocument {doc_idx}")

    for index, count in zip(doc.indices, doc.data):

        print(
            f"Word index={index}, "
            f"count={count}"
        )

# ----------------------------
# STEP 15 - Pie Chart (BoW)
# ----------------------------
top_words = words[top_indices]
top_counts = frequencies[top_indices]

plt.figure(figsize=(8,8))

plt.pie(
    top_counts,
    labels=[
        f"{w} ({c})"
        for w, c in zip(top_words, top_counts)
    ],
    autopct="%1.1f%%"
)

plt.title("Top 5 Frequent Words")
plt.show()

# ----------------------------
# STEP 16 - TF-IDF
# ----------------------------
tfidf = TfidfVectorizer(
    min_df=1,
    max_df=2
)

tfidf_matrix = tfidf.fit_transform(processed_docs)

feature_names = tfidf.get_feature_names_out()

# ----------------------------
# STEP 17 - Top TF-IDF words
# ----------------------------
for i, name in enumerate(book_names):

    row = tfidf_matrix[i].toarray()[0]

    top = row.argsort()[::-1][:5]

    labels = [feature_names[j] for j in top]
    values = [row[j] for j in top]

    print(f"\nTOP TF-IDF WORDS - {name}")

    for l, v in zip(labels, values):
        print(l, round(v,4))

    plt.figure(figsize=(8,8))

    plt.pie(
        values,
        labels=[
            f"{l} ({v:.2f})"
            for l, v in zip(labels, values)
        ],
        autopct="%1.1f%%"
    )

    plt.title(f"TF-IDF Top Words\n{name}")

    plt.show()

Question 7: Difference between Stemming and Lemmatization
Stemming

Uses simple rules to cut words.
Produces shortened forms that may not be real words.
Example:

running → run
studies → studi



Lemmatization

Uses linguistic knowledge and vocabulary.
Produces valid dictionary words.
Example:

running → run
studies → study
better → good



Conclusion: Lemmatization is usually more accurate and meaningful for NLP analysis, while stemming is faster but less precise.

Question 5: Analysis of BoW Results
You can write:

The most frequent words are often character names and common story terms.
Words such as alice, said, little, and one are expected because they appear throughout the books.
These words are not always informative because they dominate the frequency count.
BoW only considers frequency and ignores context.


TF-IDF Analysis
You can write:

TF-IDF reduces the importance of words that appear frequently in all books.
It highlights words that are more unique to a specific book.
The resulting keywords are more informative than the simple BoW frequencies.
TF-IDF helps identify the main themes and distinguishing concepts of each document.